Mount Google *Drive*

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


install libraries

In [8]:
!pip install ultralytics
!pip install roboflow

import libraries

In [9]:
import ultralytics
ultralytics.checks()

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 21.7/107.7 GB disk)


Unzip dataset

In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/AI Football Analytics System.zip"
extract_path = "/content/drive/MyDrive/football-analytics-ai/Dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset Extracted Successfully")

Dataset Extracted Successfully


In [4]:
import os

print(os.listdir("/content/drive/MyDrive/football-analytics-ai/Dataset/V1"))

['coco_test_annotations', 'coco_train_annotations', 'images', 'labels', 'samples', 'data.yaml']


In [10]:
!cat /content/drive/MyDrive/football-analytics-ai/Dataset/V1/data.yaml

names:
  0: Player
  1: Ball
  2: Referee
path: /content/drive/MyDrive/football-analytics-ai/Dataset/V1
train: images/train
val: images/test


In [ ]:
import os

train_label_dir = "/content/drive/MyDrive/football-analytics-ai/Dataset/V1/labels/train"
test_label_dir = "/content/drive/MyDrive/football-analytics-ai/Dataset/V1/labels/test"

print("Train labels:", len(os.listdir(train_label_dir)))
print("Test labels:", len(os.listdir(test_label_dir)))

# Print first label
first_label = os.listdir(train_label_dir)[0]
with open(os.path.join(train_label_dir, first_label)) as f:
    print(f"Sample content of {first_label}:\n", f.read())

Train labels: 8677
Test labels: 2996
Sample content of v7_sahi_320_7677.txt:
 


Fine tuning of model

In [ ]:
from ultralytics import YOLO
import shutil
import yaml # Added to handle YAML file modification

# Define the path to the data.yaml file
data_yaml_path = "/content/drive/MyDrive/football-analytics-ai/Dataset/V1/data.yaml"

# Load the existing data.yaml content
with open(data_yaml_path, 'r') as file:
    data_yaml = yaml.safe_load(file)

# Update the 'path' key to the correct absolute path in Colab
data_yaml['path'] = "/content/drive/MyDrive/football-analytics-ai/Dataset/V1"

# Write the updated content back to data.yaml
with open(data_yaml_path, 'w') as file:
    yaml.dump(data_yaml, file)

print(f"Updated data.yaml content: {data_yaml}")

# Load YOLOv8 pretrained model
model = YOLO("yolov8n.pt")

# Start training
model.train(
    data=data_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0
)

# Copy best model to Google Drive
shutil.copy("runs/detect/train/weights/best.pt", "/content/drive/MyDrive/football-analytics-ai/Model/best_model.pt")

print("Best model saved to Google Drive successfully ✅")

Updated data.yaml content: {'names': {0: 'Player', 1: 'Ball', 2: 'Referee'}, 'path': '/content/drive/MyDrive/football-analytics-ai/Dataset/V1', 'train': 'images/train', 'val': 'images/test'}
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/football-analytics-ai/Dataset/V1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_rat

Exception in thread Thread-14 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage_fd
    fd = df.detach()
         ^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/resource_

KeyboardInterrupt: 

In [ ]:
import shutil
import os

# Path where YOLOv8 saves checkpoints
weights_path = "runs/detect/train/weights/last.pt"

# Your Google Drive folder
backup_path = "/content/drive/MyDrive/football-analytics-ai/Model/last_epoch_1.pt"

if os.path.exists(weights_path):
    shutil.copy(weights_path, backup_path)
    print("✅ Last checkpoint (epoch 17) saved to Google Drive successfully")
else:
    print("❌ last.pt not found. Check your training path.")

In [11]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/football-analytics-ai/Model/last_epoch_18.pt")

In [12]:
video_path = "/content/drive/MyDrive/football-analytics-ai/Dataset/fb.mp4"

In [13]:
results = model.predict(
    source=video_path,
    save=True,
    project="/content/drive/MyDrive/yolo_results",
    name="test_video_result",
    conf=0.25
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/3605) /content/drive/MyDrive/football-analytics-ai/Dataset/fb.mp4: 384x640 15 Players, 1 Ball, 2 Referees, 197.4ms
video 1/1 (frame 2/3605) /content/drive/MyDrive/football-analytics-ai/Dataset/fb.mp4: 384x640 14 Players, 1 Ball, 2 Referees, 184.4ms
video 1/1 (frame 3/3605) /content/drive/MyDrive/football-analytics-ai/Dataset/fb.mp4: 384x640 13 Players, 1 Ball, 2 Referees, 152.4ms
video 1/1 (frame 4/3605) /content/drive/MyDrive/football